In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install timm
!pip install datasets
# !pip install -U huggingface_hub

In [4]:
import torch
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
import torch.optim as optim
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from collections import OrderedDict
from torch.utils.data import DataLoader
from torchvision import transforms
import shutil
from datasets import load_dataset
from PIL import Image

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [5]:
class UniformQuantizer(nn.Module):
    def __init__(self, n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8, power_of_2 = False):
        super().__init__()
        self.n_bits = int(n_bits)
        self.symmetric = bool(symmetric)
        self.channel_wise = bool(channel_wise)
        self.momentum = float(momentum)
        self.ch_axis = int(ch_axis)
        self.eps = float(eps)
        self.power_of_2 = bool(power_of_2)

        if self.symmetric:
            self.qmin = -(2 ** (self.n_bits - 1))
            self.qmax = 2 ** (self.n_bits - 1) - 1
        else:
            self.qmin = 0
            self.qmax = 2 ** self.n_bits - 1

        # safe neutral init
        self.register_buffer("scale", torch.tensor(1.0, dtype=torch.float32))
        self._calibrated = False

    def _ensure_shape(self, sample_scale):
        # ensure self.scale has the same shape as sample_scale
        if self.scale.shape != sample_scale.shape:
            with torch.no_grad():
                self.scale.resize_(sample_scale.shape)
                self.scale.copy_(torch.zeros_like(sample_scale, device=self.scale.device))
            self._calibrated = True

    def calculate_qparams(self, x):
        if self.channel_wise:
            dims = [i for i in range(x.ndim) if i != self.ch_axis]
            x_min = x.amin(dim=dims, keepdim=True)
            x_max = x.amax(dim=dims, keepdim=True)
        else:
            x_min = x.min()
            x_max = x.max()

        if self.symmetric:
            if self.power_of_2:
                x_absmax = torch.maximum(x_min.abs(), x_max.abs())
                qmax_sym = float(2 ** (self.n_bits - 1) - 1)
                scale_log2 = torch.log2(x_absmax / qmax_sym + self.eps)
                scale = torch.sign(scale_log2) * torch.ceil(torch.abs(scale_log2))
            else:
                x_absmax = torch.maximum(x_min.abs(), x_max.abs())
                qmax_sym = float(2 ** (self.n_bits - 1) - 1)
                scale = (x_absmax / qmax_sym).clamp(min=self.eps)
        else:
            denom = float(self.qmax - self.qmin)
            scale = ((x_max - x_min) / denom).clamp(min=self.eps)

        return scale

    @torch.no_grad()
    def init_from_tensor(self, x):
        """Initialize scale/zero_point from a sample tensor (e.g. weights)."""
        scale = self.calculate_qparams(x)
        self._ensure_shape(scale)
        # copy into buffers in-place
        self.scale.copy_(scale.detach().to(self.scale.device))
        self._calibrated = True

    def forward(self, x):
        # training: fake quantize; eval: dequantized floats by default
        if self.training:
            scale_b = self.calculate_qparams(x)
            if self.channel_wise:
                self._ensure_shape(scale_b)
            # EMA update (in-place)
            with torch.no_grad():
                s_new = scale_b.detach().to(self.scale.device)
                # in-place update to preserve buffer objects in state_dict
                if self.scale.shape == s_new.shape:
                    self.scale.mul_(1.0 - self.momentum).add_(self.momentum * s_new)
                else:
                    # broadcast-safe assignment
                    self.scale.copy_((1.0 - self.momentum) * self.scale + self.momentum * s_new)

            # Fake quantize (STE-like)
            if self.power_of_2:
                x_int = torch.round(x / (2 ** self.scale))
                x_int = torch.clamp(x_int, self.qmin, self.qmax)
                x_dequant = (2 ** self.scale) * (x_int)
                # final clamp to avoid inf/nan (extra safety)
                x_dequant = torch.nan_to_num(x_dequant, nan=0.0, posinf=1e6, neginf=-1e6)
                return x_dequant
            else:
                x_int = torch.round(x / (self.scale))
                x_int = torch.clamp(x_int, self.qmin, self.qmax)
                x_dequant = (self.scale) * (x_int)
                # final clamp to avoid inf/nan (extra safety)
                x_dequant = torch.nan_to_num(x_dequant, nan=0.0, posinf=1e6, neginf=-1e6)
                return x_dequant
        else:
            if self.power_of_2:
                x_int = torch.round(x / (2 ** self.scale))
                x_int = torch.clamp(x_int, self.qmin, self.qmax)
                x_dequant = (2 ** self.scale) * (x_int)
                # final clamp to avoid inf/nan (extra safety)
                x_dequant = torch.nan_to_num(x_dequant, nan=0.0, posinf=1e6, neginf=-1e6)
                return x_dequant
            else:
                x_int = torch.round(x / (self.scale))
                x_int = torch.clamp(x_int, self.qmin, self.qmax)
                x_dequant = (self.scale) * (x_int)
                # final clamp to avoid inf/nan (extra safety)
                x_dequant = torch.nan_to_num(x_dequant, nan=0.0, posinf=1e6, neginf=-1e6)
                return x_dequant

class QuantizedModule_act_only(nn.Module):
    def __init__(self, nn_module, n_bits=8, symmetric=True, channel_wise=False, power_of_2=True):
        super().__init__()
        self.nn_module = nn_module
        # activation quant: per-tensor (axis doesn't matter)
        self.act_quant = UniformQuantizer(n_bits=n_bits, symmetric=symmetric, channel_wise=channel_wise, power_of_2=power_of_2)

    def forward(self, x):
        x_q = self.act_quant(x)

        # final matmul (using float tensor, either fake-quantized or dequantized)
        out = self.nn_module(x_q)
        return out

class QuantizedModule_full(nn.Module):
  def __init__(self, nn_module, n_bits=8, symmetric=True, channel_wise_act=False, channel_wise_param=True, power_of_2_act=True, power_of_2_param=False, norm=False):
      super().__init__()
      self.nn_module = nn_module
      # activation quant: per-tensor (axis doesn't matter)
      self.act_quant = UniformQuantizer(n_bits=n_bits, symmetric=symmetric, channel_wise=channel_wise_act, power_of_2=power_of_2_act)

      self.param_quants = nn.ModuleDict()
      for name, param in nn_module.named_parameters(recurse=False):
          self.param_quants[name] = UniformQuantizer(
              n_bits=n_bits, symmetric=symmetric,
              channel_wise=channel_wise_param, ch_axis=0, power_of_2=power_of_2_param
          )

      if isinstance(nn_module, nn.BatchNorm1d):
          for name, buf in nn_module.named_buffers(recurse=False):
              if name == "num_batches_tracked":
                  self.register_buffer(name, buf, persistent=True)

  def init_param_quant(self):
        # Initialize parameter quantizers
        for name, param in self.nn_module.named_parameters(recurse=False):
            if name in self.param_quants:
                self.param_quants[name].init_from_tensor(param.data)

  def forward(self, x):
      x_q = self.act_quant(x)

      if isinstance(self.nn_module, nn.Linear):
          wq = self.param_quants["weight"](self.nn_module.weight)
          bq = self.param_quants["bias"](self.nn_module.bias) if self.nn_module.bias is not None else None
          return F.linear(x_q, wq, bq)

      if isinstance(self.nn_module, nn.BatchNorm1d):
          wq = self.param_quants["weight"](self.nn_module.weight)
          bq = self.param_quants["bias"](self.nn_module.bias)
          return F.batch_norm(x_q, self.nn_module.running_mean, self.nn_module.running_var, wq, bq,
                              self.nn_module.training,
                              self.nn_module.momentum,
                              self.nn_module.eps)

      if isinstance(self.nn_module, nn.Conv2d):
          wq = self.param_quants["weight"](self.nn_module.weight)
          bq = self.param_quants["bias"](self.nn_module.bias) if self.nn_module.bias is not None else None
          return F.conv2d(
              x_q,
              wq,
              bq,
              stride=self.nn_module.stride,
              padding=self.nn_module.padding,
              dilation=self.nn_module.dilation,
              groups=self.nn_module.groups
          )

In [6]:
class Log2Quantizer(nn.Module):
    def __init__(self, n_bits=8, clip_min=1e-12):#, ste=True):
        super().__init__()
        self.n_bits = n_bits
        self.clip_min = clip_min
        self.qmin = 0
        self.qmax = 2 ** n_bits - 1
        #self.ste = ste  # straight-through estimator flag

    def forward(self, x):
        x = torch.clamp(x, min=self.clip_min)
        log_x = -torch.log2(x)

        q = torch.round(log_x)
        q = torch.clamp(q, self.qmin, self.qmax)

        x_q = 2 ** (-q)

        return x_q

        # if self.training and self.ste:
        #     # Straight-through estimator: pass gradients as if identity
        #    return x + (x_q - x).detach()
        # else:
        #    return x_q

In [7]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, norm_layer=nn.BatchNorm1d, out_features=None, act_layer=nn.ReLU, drop=0.):  #Batch to Layer
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        #self.fc1 = QuantizedModule_full(nn.Linear(in_features, hidden_features, bias=True))
        self.fc1 = (nn.Linear(in_features, hidden_features, bias=True))
        # self.norm1 = QuantizedModule_full(norm_layer(hidden_features))
        self.norm1 = norm_layer(hidden_features)
        # self.act = QuantizedModule_act_only(act_layer())
        self.act = (act_layer())
        # self.fc2 = QuantizedModule_full(nn.Linear(hidden_features, out_features, bias=True))
        self.fc2 = (nn.Linear(hidden_features, out_features, bias=True))
        # self.norm2 = QuantizedModule_full(norm_layer(out_features))
        self.norm2 = norm_layer(out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = x.transpose(1, 2)
        x = self.norm1(x)
        x = x.transpose(1, 2)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = x.transpose(1, 2)
        x = self.norm2(x)
        x = x.transpose(1, 2)
        x = self.drop(x)
        return x


def window_partition(x, window_size):
    """
    Args:
        x: (B, H, W, C)
        window_size (int): window size

    Returns:
        windows: (num_windows*B, window_size, window_size, C)
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    """
    Args:
        windows: (num_windows*B, window_size, window_size, C)
        window_size (int): Window size
        H (int): Height of image
        W (int): Width of image

    Returns:
        x: (B, H, W, C)
    """
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class WindowAttention(nn.Module):
    """ Window based multi-head self attention (W-MSA) module with relative position bias.
    It supports both of shifted and non-shifted window.

    Args:
        dim (int): Number of input channels.
        window_size (tuple[int]): The height and width of the window.
        num_heads (int): Number of attention heads.
        qkv_bias (bool, optional):  If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set
        attn_drop (float, optional): Dropout ratio of attention weight. Default: 0.0
        proj_drop (float, optional): Dropout ratio of output. Default: 0.0
    """

    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None, attn_drop=0., proj_drop=0.):

        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        # define a parameter table of relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))  # 2*Wh-1 * 2*Ww-1, nH

        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        # get pair-wise relative position index for each token inside the window
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
        relative_coords[:, :, 0] += self.window_size[0] - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)  # Wh*Ww, Wh*Ww
        self.register_buffer("relative_position_index", relative_position_index, persistent=False)

        # self.qkv = QuantizedModule_full(nn.Linear(dim, dim * 3, bias=qkv_bias))
        self.qkv = (nn.Linear(dim, dim * 3, bias=qkv_bias))
        self.attn_drop = nn.Dropout(attn_drop)
        # self.proj = QuantizedModule_full(nn.Linear(dim, dim))
        self.proj = (nn.Linear(dim, dim))
        self.proj_drop = nn.Dropout(proj_drop)

        trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)
        # self.log_quantizer = Log2Quantizer()
        # self.quantizer1 = UniformQuantizer(n_bits=8, symmetric=True, channel_wise=False, power_of_2=True)
        # self.quantizer2 = UniformQuantizer(n_bits=8, symmetric=True, channel_wise=False, power_of_2=True)

    def forward(self, x, mask=None):
        """
        Args:
            x: input features with shape of (num_windows*B, N, C)
            mask: (0/-inf) mask with shape of (num_windows, Wh*Ww, Wh*Ww) or None
        """
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        # qkv = self.quantizer1(qkv)
        q, k, v = qkv[0], qkv[1], qkv[2]  # make torchscript happy (cannot use tensor as tuple)

        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))
        #print(attn)

        # attn = self.quantizer2(attn)
        #print(attn)

        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)  # Wh*Ww,Wh*Ww,nH
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()  # nH, Wh*Ww, Wh*Ww
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
            # attn = self.softmax(attn * math.log(2.0))
            attn = self.softmax(attn)
            #print(attn)
        else:
            # attn = self.softmax(attn * math.log(2.0))
            attn = self.softmax(attn)
            #print(attn)

        # attn = self.log_quantizer(attn)

        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class SwinTransformerBlock(nn.Module):
    """ Swin Transformer Block.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resulotion.
        num_heads (int): Number of attention heads.
        window_size (int): Window size.
        shift_size (int): Shift size for SW-MSA.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0
        act_layer (nn.Module, optional): Activation layer. Default: nn.GELU
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.BatchNorm1d
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0., drop_path=0.,
                 act_layer=nn.ReLU, norm_layer=nn.BatchNorm1d,  #Batch to Layer
                 fused_window_process=False):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(self.input_resolution) <= self.window_size:
            # if window size is larger than input resolution, we don't partition windows
            self.shift_size = 0
            self.window_size = min(self.input_resolution)
        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"

        # self.norm1 = QuantizedModule_full(norm_layer(dim))
        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)

        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        # self.norm2 = QuantizedModule_full(norm_layer(dim))
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        if self.shift_size > 0:
            # calculate attention mask for SW-MSA
            H, W = self.input_resolution
            img_mask = torch.zeros((1, H, W, 1))  # 1 H W 1
            h_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            w_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1

            mask_windows = window_partition(img_mask, self.window_size)  # nW, window_size, window_size, 1
            mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
            #print(attn_mask)
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask, persistent=False)
        self.fused_window_process = fused_window_process

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        shortcut = x
        x = x.transpose(1, 2)
        x = self.norm1(x)
        x = x.transpose(1, 2)
        x = x.view(B, H, W, C)

        # cyclic shift
        if self.shift_size > 0:
            if not self.fused_window_process:
                shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
                # partition windows
                x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
            else:
                x_windows = WindowProcess.apply(x, B, H, W, C, -self.shift_size, self.window_size)
        else:
            shifted_x = x
            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C

        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)  # nW*B, window_size*window_size, C

        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, mask=self.attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)

        # reverse cyclic shift
        if self.shift_size > 0:
            if not self.fused_window_process:
                shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
                x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
            else:
                x = WindowProcessReverse.apply(attn_windows, B, H, W, C, self.shift_size, self.window_size)
        else:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = shifted_x
        x = x.view(B, H * W, C)
        x = shortcut + self.drop_path(x)

        # FFN
        x = x.transpose(1, 2)
        x = self.norm2(x)
        x = x.transpose(1, 2)
        x = x + self.drop_path(self.mlp(x))

        return x


class PatchMerging(nn.Module):
    """ Patch Merging Layer.

    Args:
        input_resolution (tuple[int]): Resolution of input feature.
        dim (int): Number of input channels.
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.BatchNorm1d
    """

    def __init__(self, input_resolution, dim, norm_layer=nn.BatchNorm1d):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        # self.reduction = QuantizedModule_full(nn.Linear(4 * dim, 2 * dim, bias=False))
        self.reduction = (nn.Linear(4 * dim, 2 * dim, bias=False))
        # self.norm = QuantizedModule_full(norm_layer(4 * dim))
        self.norm = norm_layer(4 * dim)

    def forward(self, x):
        """
        x: B, H*W, C
        """
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"x size ({H}*{W}) are not even."

        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C
        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C
        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C
        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C
        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C

        x = x.transpose(1, 2)
        x = self.norm(x)
        x = x.transpose(1, 2)
        x = self.reduction(x)

        return x


class BasicLayer(nn.Module):
    """ A basic Swin Transformer layer for one stage.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resolution.
        depth (int): Number of blocks.
        num_heads (int): Number of attention heads.
        window_size (int): Local window size.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0
        norm_layer (nn.Module, optional): Normalization layer. Default: nn.BatchNorm1d
        downsample (nn.Module | None, optional): Downsample layer at the end of the layer. Default: None
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False.
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.BatchNorm1d, downsample=None, use_checkpoint=False,    #Batch to Layer
                 fused_window_process=False):

        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth
        self.use_checkpoint = use_checkpoint

        # build blocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim=dim, input_resolution=input_resolution,
                                 num_heads=num_heads, window_size=window_size,
                                 shift_size=0 if (i % 2 == 0) else window_size // 2,
                                 mlp_ratio=mlp_ratio,
                                 qkv_bias=qkv_bias, qk_scale=qk_scale,
                                 drop=drop, attn_drop=attn_drop,
                                 drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                                 norm_layer=norm_layer,
                                 fused_window_process=fused_window_process)
            for i in range(depth)])

        # patch merging layer
        if downsample is not None:
            self.downsample = downsample(input_resolution, dim=dim, norm_layer=norm_layer)
        else:
            self.downsample = None

    def forward(self, x):
        for blk in self.blocks:
            if self.use_checkpoint:
                x = checkpoint.checkpoint(blk, x)
            else:
                x = blk(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x


class PatchEmbed(nn.Module):
    """ Image to Patch Embedding

    Args:
        img_size (int): Image size.  Default: 224.
        patch_size (int): Patch token size. Default: 4.
        in_chans (int): Number of input image channels. Default: 3.
        embed_dim (int): Number of linear projection output channels. Default: 96.
        norm_layer (nn.Module, optional): Normalization layer. Default: None
    """

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution
        self.num_patches = patches_resolution[0] * patches_resolution[1]

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        # self.proj = QuantizedModule_full(nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size), power_of_2_act=False)
        if norm_layer is not None:
            # self.norm = QuantizedModule_full(norm_layer(embed_dim))
            self.norm = norm_layer(embed_dim)
        else:
            self.norm = None

    def forward(self, x):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x).flatten(2).transpose(1, 2)  # B Ph*Pw C
        if self.norm is not None:
            x = x.transpose(1, 2)                     # (B, C, L)
            x = self.norm(x)
            x = x.transpose(1, 2)
        return x


class SwinTransformer(nn.Module):
    """ Swin Transformer
        A PyTorch impl of : `Swin Transformer: Hierarchical Vision Transformer using Shifted Windows`  -
          https://arxiv.org/pdf/2103.14030

    Args:
        img_size (int | tuple(int)): Input image size. Default 224
        patch_size (int | tuple(int)): Patch size. Default: 4
        in_chans (int): Number of input image channels. Default: 3
        num_classes (int): Number of classes for classification head. Default: 1000
        embed_dim (int): Patch embedding dimension. Default: 96
        depths (tuple(int)): Depth of each Swin Transformer layer.
        num_heads (tuple(int)): Number of attention heads in different layers.
        window_size (int): Window size. Default: 7
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4
        qkv_bias (bool): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float): Override default qk scale of head_dim ** -0.5 if set. Default: None
        drop_rate (float): Dropout rate. Default: 0
        attn_drop_rate (float): Attention dropout rate. Default: 0
        drop_path_rate (float): Stochastic depth rate. Default: 0.1
        norm_layer (nn.Module): Normalization layer. Default: nn.BatchNorm1d.
        ape (bool): If True, add absolute position embedding to the patch embedding. Default: False
        patch_norm (bool): If True, add normalization after patch embedding. Default: True
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self, img_size=224, patch_size=4, in_chans=3, num_classes=1000,
                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],
                 window_size=7, mlp_ratio=4., qkv_bias=True, qk_scale=None,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,
                 norm_layer=nn.BatchNorm1d, ape=False, patch_norm=True,   #Batch to Layer
                 use_checkpoint=False, fused_window_process=False, **kwargs):
        super().__init__()

        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.ape = ape
        self.patch_norm = patch_norm
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim,
            norm_layer=norm_layer if self.patch_norm else None)
        num_patches = self.patch_embed.num_patches
        patches_resolution = self.patch_embed.patches_resolution
        self.patches_resolution = patches_resolution

        # absolute position embedding
        if self.ape:
            self.absolute_pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
            trunc_normal_(self.absolute_pos_embed, std=.02)

        self.pos_drop = nn.Dropout(p=drop_rate)

        # stochastic depth
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule

        # build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(dim=int(embed_dim * 2 ** i_layer),
                               input_resolution=(patches_resolution[0] // (2 ** i_layer),
                                                 patches_resolution[1] // (2 ** i_layer)),
                               depth=depths[i_layer],
                               num_heads=num_heads[i_layer],
                               window_size=window_size,
                               mlp_ratio=self.mlp_ratio,
                               qkv_bias=qkv_bias, qk_scale=qk_scale,
                               drop=drop_rate, attn_drop=attn_drop_rate,
                               drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                               norm_layer=norm_layer,
                               downsample=PatchMerging if (i_layer < self.num_layers - 1) else None,
                               use_checkpoint=use_checkpoint,
                               fused_window_process=fused_window_process)
            self.layers.append(layer)

        # self.norm = QuantizedModule_full(norm_layer(self.num_features))
        self.norm = norm_layer(self.num_features)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        # self.head = nn.Sequential(OrderedDict([("fc", QuantizedModule_full(nn.Linear(self.num_features, num_classes)))])) if num_classes > 0 else nn.Identity()
        self.head = nn.Sequential(OrderedDict([("fc", (nn.Linear(self.num_features, num_classes)))])) if num_classes > 0 else nn.Identity()
        self.quantizer = UniformQuantizer(n_bits=8, symmetric=True, channel_wise=False, power_of_2=True)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm1d): #Batch to Layer
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, QuantizedModule_full):
            m.init_param_quant()

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'absolute_pos_embed'}

    @torch.jit.ignore
    def no_weight_decay_keywords(self):
        return {'relative_position_bias_table'}

    def forward_features(self, x):
        x = self.patch_embed(x)
        if self.ape:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)

        for layer in self.layers:
            x = layer(x)

        x = x.transpose(1, 2)
        x = self.norm(x)  # B L C
        x = x.transpose(1, 2)
        x = self.avgpool(x.transpose(1, 2))  # B C 1
        x = torch.flatten(x, 1)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        # x = self.quantizer(x)
        return x

In [8]:
custom_model = SwinTransformer(
    img_size=224,
    patch_size=4,
    in_chans=3,
    num_classes=1000,
    embed_dim=96,
    depths=[2, 2, 6, 2],
    num_heads=[3, 6, 12, 24],
    ape=False,
    patch_norm=True,
    use_checkpoint=False,
    fused_window_process=False,
)

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
custom_model.to(device)

SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0): BasicLayer(
      (blocks): ModuleList(
        (0): SwinTransformerBlock(
          (norm1): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path): Identity()
          (norm2): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_feature

In [10]:
def fix_keys(input_path, output_path):
    checkpoint = torch.load(input_path, map_location="cpu")

    # Handle both raw state_dict and wrapped dict
    state_dict = checkpoint
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    suffixes = ("weight", "bias", "running_mean", "running_var")

    new_state_dict = {}
    for key, value in state_dict.items():
        parts = key.split(".")
        last = parts[-1]

        # if ("norm" in key):
        #     # Leave key unchanged
        #     new_key = key
        if last in suffixes:
            if last in ("weight", "bias") and len(parts) >= 2 and parts[-2] == "linear":
                # Replace "linear" with "nn_module"
                parts[-2] = "nn_module"
                new_key = ".".join(parts)
            elif last in ("running_mean", "running_var"):
                # Insert nn_module before running stats
                new_key = ".".join(parts[:-1] + ["nn_module", last])
            else:
                # Default: insert nn_module before suffix
                new_key = ".".join(parts[:-1] + ["nn_module", last])
        else:
            new_key = key

        new_state_dict[new_key] = value

    # Save back in the same structure
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        checkpoint["state_dict"] = new_state_dict
        torch.save(checkpoint, output_path)
    else:
        torch.save(new_state_dict, output_path)

    print(f"✅ Saved modified checkpoint to {output_path}")

In [12]:
#fix_keys("/content/drive/MyDrive/custom_model_weights6.pth", "/content/drive/MyDrive/custom_model_weights6_.pth")

# orig_state = torch.load("/content/drive/MyDrive/Weights/custom_model_weights32.pth", map_location= 'cuda' if (torch.cuda.is_available()) else 'cpu')

def strip_quant_wrappers(state_dict):
    new_sd = {}
    for k, v in state_dict.items():
        if ".nn_module." in k:
            new_k = k.replace(".nn_module.", ".")
            new_sd[new_k] = v
        # ignore quantization-only params
    return new_sd

orig_state = torch.load("/content/drive/MyDrive/custom_model_weights32.pth")
orig_state = strip_quant_wrappers(orig_state)

missing, unexpected = custom_model.load_state_dict(orig_state, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: ['layers.0.blocks.0.attn.relative_position_bias_table', 'layers.0.blocks.1.attn.relative_position_bias_table', 'layers.1.blocks.0.attn.relative_position_bias_table', 'layers.1.blocks.1.attn.relative_position_bias_table', 'layers.2.blocks.0.attn.relative_position_bias_table', 'layers.2.blocks.1.attn.relative_position_bias_table', 'layers.2.blocks.2.attn.relative_position_bias_table', 'layers.2.blocks.3.attn.relative_position_bias_table', 'layers.2.blocks.4.attn.relative_position_bias_table', 'layers.2.blocks.5.attn.relative_position_bias_table', 'layers.3.blocks.0.attn.relative_position_bias_table', 'layers.3.blocks.1.attn.relative_position_bias_table', 'quantizer.scale']
Unexpected keys: []


In [ ]:
# def freeze_selected_params(model: nn.Module, substrings=None):
#     """
#     Freezes parameters in the model whose names contain any of the given substrings.

#     Args:
#         model (nn.Module): The PyTorch model.
#         substrings (list of str): Substrings to match in parameter names.
#     """
#     if substrings is None:
#         substrings = [
#             "act_quant.scale",
#             "param_quants.weight.scale",
#             "param_quants.bias.scale",
#             "norm",
#             "attn",
#             "mlp"
#         ]

#     for name, param in model.named_parameters():
#         if not any(s in name for s in substrings):
#             param.requires_grad = False
#             print(f"FROZEN: {name}")
#         else:
#             print(f"TRAINABLE: {name}")

#     return model

# def freeze_selected_params(model):
#     for name, param in model.named_parameters():
#         if (
#             "head" in name or
#             "act_quant" in name or
#             "param_quants" in name
#         ):
#             param.requires_grad = True
#             print(f"TRAINABLE: {name}")
#         else:
#             param.requires_grad = False
#             print(f"FROZEN: {name}")
#     return model

# def freeze_bn_stats(model):
#     for m in model.modules():
#         if isinstance(m, nn.BatchNorm1d):
#             m.eval()
#             m.requires_grad_(False)

# freeze_bn_stats(custom_model)
# custom_model = freeze_selected_params(custom_model)

In [ ]:
!hf auth login

In [ ]:
# ----------------------------------------------------------------------------
# 1) Load ImageNet-1k training set (streaming mode)
# ----------------------------------------------------------------------------
hf_dataset_train = load_dataset("ILSVRC/imagenet-1k", split="train", streaming=True)
hf_dataset_val = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True)


train_stream = hf_dataset_train.take(100_000)
val_stream   = hf_dataset_train.skip(100_000).take(15_000)
# train_stream = hf_dataset_train.shuffle(buffer_size=10_000, seed=42).take(100_000)
# val_stream   = hf_dataset_train.shuffle(buffer_size=10_000, seed=123).take(15_000)
test_stream = hf_dataset_val.take(10_000)


transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def collate_fn(batch):
    """Convert HF samples into tensors"""
    images, labels = [], []
    for sample in batch:
        img = sample["image"]
        if not isinstance(img, Image.Image):  # if numpy array
            img = Image.fromarray(img)
        img = img.convert("RGB")  # <-- ensure 3 channels
        images.append(transform(img))
        labels.append(sample["label"])
    return torch.stack(images), torch.tensor(labels)


train_loader = DataLoader(train_stream, batch_size=32, collate_fn=collate_fn)
val_loader   = DataLoader(val_stream,   batch_size=32, collate_fn=collate_fn)
test_loader = DataLoader(test_stream, batch_size=32, collate_fn=collate_fn)


def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for i, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)
        # torch.autograd.set_detect_anomaly(True)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (i + 1) % 50 == 0:
            print(f"[Epoch {epoch} | Step {i+1}] "
                  f"Loss: {running_loss/(i+1):.4f} | "
                  f"Acc: {100.*correct/total:.2f}%")

    return running_loss / (i+1), 100.*correct/total

def evaluate(model, dataloader, criterion, device):
    model.eval()
    top1, top5, total, total_loss = 0, 0, 0, 0.0

    with torch.no_grad():
        for i, (images, labels) in enumerate(dataloader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Top-1
            _, pred1 = outputs.topk(1, dim=1)
            top1 += (pred1.squeeze() == labels).sum().item()

            # Top-5
            _, pred5 = outputs.topk(5, dim=1)
            top5 += sum([labels[j].item() in pred5[j] for j in range(labels.size(0))])

            total += labels.size(0)

            if (i+1) % 50 == 0:
                print(f"[Val Step {i+1}] "
                      f"Loss: {total_loss/(i+1):.4f} | "
                      f"Top-1: {100.*top1/total:.2f}% | "
                      f"Top-5: {100.*top5/total:.2f}%")

    return 100.*top1/total, 100.*top5/total, total_loss/(i+1)

criterion = nn.CrossEntropyLoss()

device = "cuda" if torch.cuda.is_available() else "cpu"
custom_model.to(device)

In [ ]:
# optimizer = optim.Adam(custom_model.parameters(), lr=1e-5)
optimizer = optim.AdamW([
    {"params": custom_model.head.parameters(), "lr": 3e-4},
    {"params": [p for n,p in custom_model.named_parameters()
                if "head" not in n and p.requires_grad], "lr": 1e-5}
], weight_decay=1e-4)

In [ ]:
EPOCHS = 1  # just for demo, set higher for real training
for epoch in range(1, EPOCHS+1):
    train_loss, train_acc = train_one_epoch(custom_model, train_loader, criterion, optimizer, device, epoch)
    print(f"\n[Epoch {epoch}] Training Loss: {train_loss:.4f} | Training Acc: {train_acc:.2f}%")

    val_top1, val_top5, val_loss = evaluate(custom_model, val_loader, criterion, device)
    print(f"[Epoch {epoch}] Validation → Loss: {val_loss:.4f} | Top-1: {val_top1:.2f}% | Top-5: {val_top5:.2f}%\n")

In [ ]:
# accuracy after replacing gelu by relu
top1, top5, avg_loss = evaluate(custom_model, test_loader, criterion, device)
print(f"\nValidation Accuracy → Top-1: {top1:.2f}% | Top-5: {top5:.2f}% | Loss: {avg_loss:.4f}")

In [ ]:
#save your work to your drive
torch.save(custom_model.state_dict(), "custom_model_weights34.pth")

drive_path = "/content/drive/MyDrive/Weights/custom_model_weights34.pth"
shutil.copy("custom_model_weights34.pth", drive_path)

In [ ]:
orig_state = torch.load("/content/drive/MyDrive/Weights/custom_model_weights34.pth")
with open("weights34.txt", "w") as f:
    for name, param in orig_state.items():
        f.write(f"{name}: {param}\n\n")

drive_path = "/content/drive/MyDrive/Weights/weights34.txt"
shutil.copy("weights34.txt", drive_path)

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()